# FAVED Group 4: Pest Detection (YOLO11), Evaluation and Unified Pest + Disease Pipeline
**Covers Team 2 (training) + Team 3 (evaluation, error analysis, integration, RAG hand-off).**
Crops: **maize (5 pests) + cocoa (4 pests)**, built so more crops can be added (Section 9).

| Section | What it does |
|---|---|
| 0 | Setup, Drive paths, GPU check |
| 1-2 | Load Team 1 dataset, **audit** it (leakage, duplicates, balance), rebuild cocoa splits leak-safe |
| 3 | Multi-experiment YOLO11 training (resumable on Colab disconnects) + comparison table |
| 4 | Select best model -> `pest_best.pt` + model card |
| 5 | Test-set evaluation: P, R, F1, mAP@50, mAP@50-95, confusion matrix, per-crop |
| 6 | Error analysis: TP / FP / FN, confusion pairs, galleries |
| 7 | Robustness (phone-photo conditions) + unseen real-world images |
| 8 | Review FAVED disease model, **unified pest+disease output**, RAG hand-off |
| 9 | Add new crops later (class registry) |
| 10 | Auto-generated report |

**Before running:** put `Combined_Pest_Dataset.zip` (Team 1 output) in Drive at `MyDrive/FAVED_Pest_Group4/`. Team 1 only saved it in `/content`, which is wiped when the runtime resets.

## 0. Setup

In [2]:
!pip install -q "ultralytics==8.4.155" pyyaml scikit-learn tabulate
import os, re, json, shutil, random, time, sys
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd, yaml, cv2, torch
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/FAVED_Pest_Group4"); ROOT.mkdir(parents=True, exist_ok=True)
RUNS, FINAL, EVAL = ROOT/"runs", ROOT/"final", ROOT/"evaluation"
for d in (RUNS, FINAL, EVAL): d.mkdir(exist_ok=True)

DATASET_ZIP         = ROOT/"Combined_Pest_Dataset.zip"        # Team 1 output
UNSEEN_DIR          = ROOT/"unseen_images"                    # real farm photos (no labels needed)
FAVED_DISEASE_MODEL = ROOT/"faved_disease_model.pt"           # DETECTION_MODEL_PATH from FAVED backend
UNSEEN_DIR.mkdir(exist_ok=True)

DATA_SRC  = Path("/content/Pest_Data_src")   # Team 1 as delivered
DATA      = Path("/content/Pest_Data")       # audited + fixed dataset used for training
DATA_YAML = DATA/"data.yaml"
SPLITS    = ["train", "valid", "test"]
DEVICE    = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == 0 else "NONE (switch runtime to GPU)")

NAMES = ["Large cutworm","Yellow cutworm","Corn borer","Army worm","Aphids",
         "Pod-borer","Weevil","Mirid-bug","Mealybug"]
CROP_OF = {0:"maize",1:"maize",2:"maize",3:"maize",4:"maize",5:"cocoa",6:"cocoa",7:"cocoa",8:"cocoa"}
IMG_EXT = {".jpg",".jpeg",".png"}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Mounted at /content/drive
GPU: Tesla T4


## 1. Load Team 1 dataset
Maize = IP102 classes 19, 20, 22, 23, 24 (class IDs 0-4). Cocoa = CacaoAID pests (IDs 5-8: pod-borer, weevil, mirid-bug, mealybug).
Note: Team 1 **did** include cocoa. The problems are in how it was split and checked (next section).

In [3]:
if not DATA_SRC.exists():
    assert DATASET_ZIP.exists(), f"Upload Team 1 zip to {DATASET_ZIP}"
    shutil.unpack_archive(str(DATASET_ZIP), str(DATA_SRC))
print(open(DATA_SRC/"data.yaml").read())

path: /content/Combined_Pest_Dataset
train: train/images
val: valid/images
test: test/images

names:
  0: Large cutworm
  1: Yellow cutworm
  2: Corn borer
  3: Army worm
  4: Aphids
  5: Pod-borer
  6: Weevil
  7: Mirid-bug
  8: Mealybug


## 2. Dataset audit (Team 1 hand-off check)
Checks: class balance per split/crop, **source-image leakage** (cocoa files are Roboflow augmentations: `name_jpg.rf.<hash>`), and **exact duplicate images across splits**.
Team 1's leakage check used the regex `cocoa_\d+_jpg`, which skips names like `cocoa_IMG_7900_jpg...`, so part of the cocoa set was never checked. This audit uses a general key.

In [4]:
def source_key(stem):
    m = re.match(r"^(.*?)_jpg\.rf\.[0-9a-f]+$", stem)       # Roboflow augmented variant -> original name
    return m.group(1) if m else stem

def label_df(root):
    rows = []
    for s in SPLITS:
        for lf in (root/s/"labels").glob("*.txt"):
            crop = "cocoa" if lf.name.startswith("cocoa_") else "maize"
            cls = [int(float(l.split()[0])) for l in lf.read_text().splitlines() if l.strip()] or [-1]
            for c in cls: rows.append(dict(split=s, crop=crop, file=lf.stem, cls=c))
    return pd.DataFrame(rows)

def dhash(path, size=8):
    a = np.asarray(Image.open(path).convert("L").resize((size+1, size), Image.LANCZOS), dtype=np.int16)
    return int("".join("1" if x else "0" for x in (a[:,1:] > a[:,:-1]).flatten()), 2)

def audit(root, tag):
    df = label_df(root)
    print(f"===== AUDIT: {tag} =====")
    print("\nImages per split x crop"); print(df.drop_duplicates("file").pivot_table(index="crop", columns="split", values="file", aggfunc="count", fill_value=0)[SPLITS])
    inst = df[df.cls >= 0].pivot_table(index="cls", columns="split", values="file", aggfunc="count", fill_value=0)[SPLITS]
    inst.index = [f"{c} {NAMES[c]}" for c in inst.index]; print("\nInstances per class x split"); print(inst)
    keys = defaultdict(set)
    for s in SPLITS:
        for lf in (root/s/"labels").glob("cocoa_*.txt"): keys[s].add(source_key(lf.stem))
    print("\nCocoa unique source images:", {s: len(k) for s, k in keys.items()})
    print("Cocoa source overlap train&valid:", len(keys["train"]&keys["valid"]), "| train&test:", len(keys["train"]&keys["test"]), "| valid&test:", len(keys["valid"]&keys["test"]))
    h = defaultdict(set)
    for s in SPLITS:
        for p in (root/s/"images").iterdir():
            if p.suffix.lower() in IMG_EXT: h[dhash(p)].add(s)
    dup = sum(1 for v in h.values() if len(v) > 1)
    print("Identical images appearing in >1 split (dHash):", dup)
    return df
before = audit(DATA_SRC, "Team 1 as delivered")

===== AUDIT: Team 1 as delivered =====

Images per split x crop
split  train  valid  test
crop                     
cocoa   1731    166    82
maize   1815    277   279

Instances per class x split
split             train  valid  test
0 Large cutworm     339     15    32
1 Yellow cutworm    321     31    32
2 Corn borer        311     79    69
3 Army worm         312     32    27
4 Aphids            984    201   195
5 Pod-borer         429     40    21
6 Weevil            465     56    20
7 Mirid-bug         432     43    24
8 Mealybug          432     46    25

Cocoa unique source images: {'train': 587, 'valid': 166, 'test': 82}
Cocoa source overlap train&valid: 0 | train&test: 0 | valid&test: 0
Identical images appearing in >1 split (dHash): 44


### 2b. Rebuild cocoa splits without leakage
- Group cocoa files by **source image**, then split groups 70/15/15, stratified by dominant class.
- **Train** keeps all augmented variants. **Valid/test keep 1 variant per source**, so scores are not inflated by near-copies.
- Maize is copied unchanged (Team 1's maize split was already by distinct image; the augmented copies exist only in train).

In [5]:
def rebuild_dataset(src, dst, seed=42, ratios=(0.70, 0.15, 0.15)):
    if dst.exists(): shutil.rmtree(dst)
    for s in SPLITS:
        (dst/s/"images").mkdir(parents=True); (dst/s/"labels").mkdir(parents=True)
    groups = defaultdict(list)
    for s in SPLITS:
        for img in sorted((src/s/"images").iterdir()):
            if img.suffix.lower() not in IMG_EXT: continue
            lbl = src/s/"labels"/(img.stem + ".txt")
            if not img.name.startswith("cocoa_"):
                shutil.copy2(img, dst/s/"images"/img.name); shutil.copy2(lbl, dst/s/"labels"/lbl.name)
            else:
                groups[source_key(img.stem)].append((img, lbl))
    def dominant(lbl):
        c = Counter(int(float(l.split()[0])) for l in lbl.read_text().splitlines() if l.strip())
        return c.most_common(1)[0][0] if c else -1
    by_cls = defaultdict(list)
    for k, v in groups.items(): by_cls[dominant(v[0][1])].append(k)
    rng, assign = random.Random(seed), {}
    for c, keys in by_cls.items():
        keys.sort(); rng.shuffle(keys); n = len(keys); a = int(n*ratios[0]); b = a + int(n*ratios[1])
        for i, k in enumerate(keys): assign[k] = "train" if i < a else "valid" if i < b else "test"
    for k, items in groups.items():
        sp = assign[k]
        for img, lbl in (items if sp == "train" else items[:1]):
            shutil.copy2(img, dst/sp/"images"/img.name); shutil.copy2(lbl, dst/sp/"labels"/lbl.name)
    cfg = {"path": str(dst), "train": "train/images", "val": "valid/images", "test": "test/images",
           "nc": len(NAMES), "names": {i: n for i, n in enumerate(NAMES)}}
    (dst/"data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

rebuild_dataset(DATA_SRC, DATA)
after = audit(DATA, "After leak-safe rebuild")
shutil.make_archive(str(ROOT/"Combined_Pest_Dataset_v2"), "zip", DATA)   # persist the fixed dataset

===== AUDIT: After leak-safe rebuild =====

Images per split x crop
split  train  valid  test
crop                     
cocoa   1398    123   130
maize   1815    277   279

Instances per class x split
split             train  valid  test
0 Large cutworm     339     15    32
1 Yellow cutworm    321     31    32
2 Corn borer        311     79    69
3 Army worm         312     32    27
4 Aphids            984    201   195
5 Pod-borer         334     31    35
6 Weevil            379     37    39
7 Mirid-bug         357     30    32
8 Mealybug          354     35    36

Cocoa unique source images: {'train': 582, 'valid': 123, 'test': 130}
Cocoa source overlap train&valid: 0 | train&test: 0 | valid&test: 0
Identical images appearing in >1 split (dHash): 37


'/content/drive/MyDrive/FAVED_Pest_Group4/Combined_Pest_Dataset_v2.zip'

## 3. YOLO11 training experiments (Team 2)
- Transfer learning from COCO-pretrained YOLO11 weights, **trained on Team 1's real bounding boxes**. (The earlier Team 2 notebook trained on the IP102 *classification* folders with a fake full-image box `0.5 0.5 1.0 1.0`. That makes it a classifier in disguise, and mAP@50 ~ mAP@50-95 is the telltale sign. It also never used the maize+cocoa dataset.)
- Three experiments: nano baseline, small baseline, small + tuned hyper-parameters. Toggle `run` in the list; a 4th (medium, larger image) is off by default.
- Resumable: each run writes a `DONE` marker; interrupted runs continue from `last.pt`.
- Selection uses **validation** mAP@50-95. Final numbers are reported on the **test** split only (Section 5).

In [ ]:
EXPERIMENTS = [
  dict(name="y11n_baseline", model="yolo11n.pt", run=True,  epochs=60, imgsz=640, batch=16, extra={}),
  dict(name="y11s_baseline", model="yolo11s.pt", run=True,  epochs=60, imgsz=640, batch=16, extra={}),
  dict(name="y11s_tuned",    model="yolo11s.pt", run=True,  epochs=60, imgsz=640, batch=16,
       extra=dict(lr0=0.005, cos_lr=True, mosaic=1.0, close_mosaic=10, hsv_h=0.02, hsv_s=0.6, hsv_v=0.4,
                  degrees=10, scale=0.6, fliplr=0.5, flipud=0.0, mixup=0.1)),
  dict(name="y11m_imgsz800", model="yolo11m.pt", run=False, epochs=60, imgsz=800, batch=8, extra=dict(cos_lr=True)),
]

def run_experiment(cfg):
    run_dir = RUNS/cfg["name"]; last = run_dir/"weights"/"last.pt"; done = run_dir/"DONE"
    if done.exists(): print("skip (done):", cfg["name"]); return
    if last.exists():
        print("resuming:", cfg["name"]); YOLO(str(last)).train(resume=True)
    else:
        print("training:", cfg["name"])
        YOLO(cfg["model"]).train(data=str(DATA_YAML), epochs=cfg["epochs"], imgsz=cfg["imgsz"], batch=cfg["batch"],
                                 device=DEVICE, workers=2, patience=15, seed=0, deterministic=True, pretrained=True,
                                 project=str(RUNS), name=cfg["name"], exist_ok=True, plots=True, **cfg["extra"])
    done.write_text(time.ctime())

for cfg in EXPERIMENTS:
    if cfg["run"]: run_experiment(cfg)

skip (done): y11n_baseline
resuming: y11s_baseline
New https://pypi.org/project/ultralytics/8.4.171 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Pest_Data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4,

In [ ]:
rows = []
for cfg in EXPERIMENTS:
    csv = RUNS/cfg["name"]/"results.csv"
    if not csv.exists(): continue
    d = pd.read_csv(csv); d.columns = [c.strip() for c in d.columns]
    b = d.loc[d["metrics/mAP50-95(B)"].idxmax()]
    rows.append(dict(experiment=cfg["name"], model=cfg["model"], imgsz=cfg["imgsz"], best_epoch=int(b["epoch"]), epochs_run=len(d),
                     val_P=b["metrics/precision(B)"], val_R=b["metrics/recall(B)"],
                     val_mAP50=b["metrics/mAP50(B)"], val_mAP50_95=b["metrics/mAP50-95(B)"]))
comp = pd.DataFrame(rows).sort_values("val_mAP50_95", ascending=False).reset_index(drop=True)
comp.to_csv(EVAL/"experiment_comparison.csv", index=False); display(comp.round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for cfg in EXPERIMENTS:
    csv = RUNS/cfg["name"]/"results.csv"
    if not csv.exists(): continue
    d = pd.read_csv(csv); d.columns = [c.strip() for c in d.columns]
    ax[0].plot(d["epoch"], d["metrics/mAP50-95(B)"], label=cfg["name"]); ax[1].plot(d["epoch"], d["val/cls_loss"], label=cfg["name"])
ax[0].set_title("Val mAP@50-95"); ax[1].set_title("Val cls loss"); ax[0].legend(); plt.tight_layout(); plt.savefig(EVAL/"training_curves.png", dpi=120); plt.show()

## 4. Select the best model, save checkpoint + model card

In [ ]:
best_name = comp.iloc[0]["experiment"]
BEST_SRC = RUNS/best_name/"weights"/"best.pt"
shutil.copy2(BEST_SRC, FINAL/"pest_best.pt"); shutil.copy2(DATA_YAML, FINAL/"data.yaml")
card = dict(model="YOLO11", selected_run=best_name, selected_by="val mAP50-95", classes=NAMES,
            crop_of_class={NAMES[i]: c for i, c in CROP_OF.items()}, imgsz=int(comp.iloc[0]["imgsz"]),
            val_metrics={k: float(comp.iloc[0][k]) for k in ["val_P","val_R","val_mAP50","val_mAP50_95"]},
            dataset="Combined_Pest_Dataset_v2 (IP102 maize pests + CacaoAID cocoa pests, leak-safe cocoa split)")
(FINAL/"model_card.json").write_text(json.dumps(card, indent=2)); print(json.dumps(card, indent=2))
model = YOLO(str(FINAL/"pest_best.pt"))

## 5. Evaluation on the held-out TEST split (Team 3)
Precision, Recall, F1, mAP@50, mAP@50-95 (overall, per class, per crop), confusion matrix, and the F1-optimal confidence threshold.

In [ ]:
m = model.val(data=str(DATA_YAML), split="test", imgsz=int(card["imgsz"]), conf=0.001, iou=0.6, plots=True,
              project=str(EVAL), name="test_overall", exist_ok=True, device=DEVICE)
P, R = float(m.box.mp), float(m.box.mr)
overall = dict(Precision=P, Recall=R, F1=2*P*R/(P+R+1e-9), mAP50=float(m.box.map50), mAP50_95=float(m.box.map))
print("OVERALL (test):", {k: round(v, 4) for k, v in overall.items()})

rows = []
for i, ci in enumerate(m.box.ap_class_index):
    ci = int(ci); p, r = float(m.box.p[i]), float(m.box.r[i])
    rows.append(dict(class_id=ci, name=NAMES[ci], crop=CROP_OF[ci], P=p, R=r, F1=2*p*r/(p+r+1e-9),
                     mAP50=float(m.box.ap50[i]), mAP50_95=float(m.box.ap[i])))
per_class = pd.DataFrame(rows); per_class.to_csv(EVAL/"per_class_metrics.csv", index=False); display(per_class.round(3))
display(per_class.groupby("crop")[["P","R","F1","mAP50","mAP50_95"]].mean().round(3))

OPT_CONF = 0.25
try:
    f1c, px = np.array(m.box.f1_curve), np.array(m.box.px)
    OPT_CONF = float(px[f1c.mean(0).argmax()]); print("F1-optimal confidence threshold:", round(OPT_CONF, 3))
except Exception as e: print("f1_curve unavailable, using 0.25:", e)

for f in ["confusion_matrix_normalized.png", "F1_curve.png", "PR_curve.png"]:
    p = EVAL/"test_overall"/f
    if p.exists(): plt.figure(figsize=(8, 7)); plt.imshow(plt.imread(p)); plt.axis("off"); plt.title(f); plt.show()

In [ ]:
# Per-crop test metrics (does adding cocoa hurt maize or vice versa?)
def val_on_files(files, tag, root=EVAL/"tmp_lists"):
    root.mkdir(parents=True, exist_ok=True)
    lst = root/f"{tag}.txt"; lst.write_text("\n".join(str(f) for f in files))
    y = root/f"{tag}.yaml"; y.write_text(yaml.safe_dump({"path": str(DATA), "train": str(lst), "val": str(lst), "names": {i: n for i, n in enumerate(NAMES)}}))
    r = model.val(data=str(y), imgsz=int(card["imgsz"]), conf=0.001, iou=0.6, plots=False, verbose=False, device=DEVICE, project=str(EVAL), name=f"tmp_{tag}", exist_ok=True)
    p, rc = float(r.box.mp), float(r.box.mr)
    return dict(subset=tag, images=len(files), P=p, R=rc, F1=2*p*rc/(p+rc+1e-9), mAP50=float(r.box.map50), mAP50_95=float(r.box.map))
test_imgs = sorted(p for p in (DATA/"test"/"images").iterdir() if p.suffix.lower() in IMG_EXT)
crop_tbl = pd.DataFrame([val_on_files([p for p in test_imgs if p.name.startswith("cocoa_")], "cocoa_only"),
                         val_on_files([p for p in test_imgs if not p.name.startswith("cocoa_")], "maize_only")])
crop_tbl.to_csv(EVAL/"per_crop_metrics.csv", index=False); display(crop_tbl.round(3))

## 6. Error analysis: false positives and false negatives
Each test image is predicted at the F1-optimal threshold and matched to ground truth (IoU >= 0.5, greedy by confidence).
- **TP**: right class, right place
- **FP / class_confusion**: box on a real object, wrong class
- **FP / localization**: right class, box too loose or shifted (0.1 <= IoU < 0.5)
- **FP / background**: box where there is no labelled object
- **FN / misclassified**: real pest predicted as another class
- **FN / missed**: nothing predicted

In [ ]:
def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:,None,0], b[None,:,0]); y1 = np.maximum(a[:,None,1], b[None,:,1])
    x2 = np.minimum(a[:,None,2], b[None,:,2]); y2 = np.minimum(a[:,None,3], b[None,:,3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    aa = (a[:,2]-a[:,0])*(a[:,3]-a[:,1]); bb = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return inter / (aa[:,None] + bb[None,:] - inter + 1e-9)

def read_gt(lbl, w, h):
    cls, box = [], []
    for l in lbl.read_text().splitlines():
        p = l.split()
        if len(p) < 5: continue
        c, xc, yc, bw, bh = int(float(p[0])), *map(float, p[1:5])
        cls.append(c); box.append([(xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h])
    return np.array(cls, int), np.array(box, float).reshape(-1, 4)

def analyze_errors(model, imgs, conf, iou_thr=0.5):
    ev = []
    for img in imgs:
        r = model.predict(str(img), conf=conf, iou=0.6, imgsz=int(card["imgsz"]), verbose=False, device=DEVICE)[0]
        h, w = r.orig_shape
        pb, pc, ps = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()
        gc, gb = read_gt(DATA/"test"/"labels"/(img.stem + ".txt"), w, h)
        M = iou_mat(pb, gb); used_g, used_p = set(), set()
        for p in np.argsort(-ps):
            cand = [(M[p, g], g) for g in range(len(gc)) if g not in used_g and gc[g] == pc[p] and M[p, g] >= iou_thr]
            if cand:
                _, g = max(cand); used_g.add(g); used_p.add(p)
                ev.append(dict(image=img.name, kind="TP", subtype="ok", true=gc[g], pred=pc[p], conf=ps[p], iou=M[p, g]))
        for p in range(len(pc)):
            if p in used_p: continue
            j = int(M[p].argmax()) if len(gc) else -1; best = M[p, j] if j >= 0 else 0
            if best >= iou_thr and gc[j] != pc[p]: sub, tr = "class_confusion", gc[j]
            elif best >= 0.1 and gc[j] == pc[p]:   sub, tr = "localization", gc[j]
            else:                                  sub, tr = "background", -1
            ev.append(dict(image=img.name, kind="FP", subtype=sub, true=tr, pred=pc[p], conf=ps[p], iou=best))
        for g in range(len(gc)):
            if g in used_g: continue
            j = int(M[:, g].argmax()) if len(pc) else -1; best = M[j, g] if j >= 0 else 0
            if best >= iou_thr: ev.append(dict(image=img.name, kind="FN", subtype="misclassified", true=gc[g], pred=pc[j], conf=ps[j], iou=best))
            else:               ev.append(dict(image=img.name, kind="FN", subtype="missed", true=gc[g], pred=-1, conf=0.0, iou=best))
    return pd.DataFrame(ev)

ev = analyze_errors(model, test_imgs, OPT_CONF); ev.to_csv(EVAL/"error_events.csv", index=False)
print(f"Threshold {OPT_CONF:.2f}"); display(ev.groupby(["kind", "subtype"]).size().rename("count").to_frame())

rows = []
for c in range(len(NAMES)):
    tp = ((ev.kind == "TP") & (ev.true == c)).sum(); fn = ((ev.kind == "FN") & (ev.true == c)).sum(); fp = ((ev.kind == "FP") & (ev.pred == c)).sum()
    p, r = tp/(tp+fp+1e-9), tp/(tp+fn+1e-9)
    rows.append(dict(class_id=c, name=NAMES[c], crop=CROP_OF[c], TP=tp, FP=fp, FN=fn, precision=p, recall=r, F1=2*p*r/(p+r+1e-9)))
err_tbl = pd.DataFrame(rows); err_tbl.to_csv(EVAL/"error_by_class.csv", index=False); display(err_tbl.round(3))

conf_pairs = Counter((NAMES[int(r.true)], NAMES[int(r.pred)]) for r in ev[(ev.subtype.isin(["class_confusion", "misclassified"]))].itertuples())
print("\nMost common true -> predicted confusions:"); [print(f"  {a} -> {b}: {n}") for (a, b), n in conf_pairs.most_common(8)]
cross = sum(n for (a, b), n in conf_pairs.items() if CROP_OF[NAMES.index(a)] != CROP_OF[NAMES.index(b)])
print("Cross-crop confusions (maize <-> cocoa):", cross)

In [ ]:
# Galleries: worst images (most FP+FN). Green = ground truth, red = prediction
def draw(img_path, conf):
    im = cv2.imread(str(img_path)); h, w = im.shape[:2]
    gc, gb = read_gt(DATA/"test"/"labels"/(img_path.stem + ".txt"), w, h)
    for c, b in zip(gc, gb): cv2.rectangle(im, tuple(map(int, b[:2])), tuple(map(int, b[2:])), (0, 200, 0), 3); cv2.putText(im, "GT " + NAMES[c], (int(b[0]), max(20, int(b[1])-6)), 0, 0.7, (0, 200, 0), 2)
    r = model.predict(str(img_path), conf=conf, iou=0.6, imgsz=int(card["imgsz"]), verbose=False, device=DEVICE)[0]
    for b, c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()):
        cv2.rectangle(im, tuple(map(int, b[:2])), tuple(map(int, b[2:])), (0, 0, 255), 2); cv2.putText(im, f"{NAMES[c]} {s:.2f}", (int(b[0]), int(b[3])+22), 0, 0.7, (0, 0, 255), 2)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def gallery(names, title, out):
    names = list(names)[:12]
    if not names: print("none:", title); return
    cols = 4; rows_ = (len(names) + cols - 1) // cols
    fig, axs = plt.subplots(rows_, cols, figsize=(5*cols, 4*rows_)); axs = np.array(axs).reshape(-1)
    for a in axs: a.axis("off")
    for a, n in zip(axs, names): a.imshow(draw(DATA/"test"/"images"/n, OPT_CONF)); a.set_title(n[:28], fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(EVAL/out, dpi=90); plt.show()

bad = ev[ev.kind.isin(["FP", "FN"])].groupby("image").size().sort_values(ascending=False)
gallery(bad.index, "Worst test images (most errors)", "gallery_worst.png")
gallery(ev[(ev.kind == "FN") & (ev.subtype == "missed")].image.unique(), "False negatives: missed pests", "gallery_fn_missed.png")
gallery(ev[(ev.kind == "FP") & (ev.subtype == "background")].image.unique(), "False positives: background", "gallery_fp_background.png")

## 7. Real-world checks
**7a. Robustness**: the test set is cleaner than a farmer's phone photo. We degrade test images (dark, glare, blur, low-res/JPEG, noise) and re-score. A big drop means the model needs field data or stronger augmentation.
**7b. Unseen images**: drop real farm photos into `unseen_images/` on Drive. Without labels this is only a qualitative check. For real-world *metrics*, label 50+ field photos per class and add them as a separate test set.

In [ ]:
def corrupt(img, kind):
    if kind == "dark":   return np.clip(img * 0.45, 0, 255).astype(np.uint8)
    if kind == "glare":  return cv2.convertScaleAbs(img, alpha=1.35, beta=50)
    if kind == "blur":   return cv2.GaussianBlur(img, (11, 11), 0)
    if kind == "noise":  return np.clip(img + np.random.normal(0, 18, img.shape), 0, 255).astype(np.uint8)
    if kind == "lowres":
        h, w = img.shape[:2]; s = cv2.resize(img, (max(32, w//4), max(32, h//4))); s = cv2.resize(s, (w, h))
        ok, buf = cv2.imencode(".jpg", s, [cv2.IMWRITE_JPEG_QUALITY, 25]); return cv2.imdecode(buf, 1)

res = [dict(condition="clean", **{k: v for k, v in val_on_files(test_imgs, "clean").items() if k in ("mAP50", "mAP50_95", "R")})]
np.random.seed(0)
for kind in ["dark", "glare", "blur", "noise", "lowres"]:
    d = Path(f"/content/robust/{kind}"); shutil.rmtree(d, ignore_errors=True); (d/"images").mkdir(parents=True); (d/"labels").mkdir()
    for p in test_imgs:
        cv2.imwrite(str(d/"images"/(p.stem + ".jpg")), corrupt(cv2.imread(str(p)), kind)); shutil.copy2(DATA/"test"/"labels"/(p.stem + ".txt"), d/"labels")
    r = val_on_files(sorted((d/"images").glob("*.jpg")), kind); res.append(dict(condition=kind, mAP50=r["mAP50"], mAP50_95=r["mAP50_95"], R=r["R"]))
rob = pd.DataFrame(res); rob.to_csv(EVAL/"robustness.csv", index=False); display(rob.round(3))
rob.set_index("condition")[["mAP50", "mAP50_95"]].plot.bar(figsize=(8, 3.5), title="Robustness (test split)"); plt.tight_layout(); plt.savefig(EVAL/"robustness.png", dpi=110); plt.show()

In [ ]:
unseen = sorted(p for p in UNSEEN_DIR.iterdir() if p.suffix.lower() in IMG_EXT)
print(len(unseen), "unseen images")
if unseen:
    out = EVAL/"unseen"; out.mkdir(exist_ok=True); recs = []
    for p in unseen:
        r = model.predict(str(p), conf=OPT_CONF, iou=0.6, imgsz=int(card["imgsz"]), verbose=False, device=DEVICE)[0]
        cv2.imwrite(str(out/p.name), r.plot())
        recs.append(dict(image=p.name, detections=[(NAMES[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)]))
    pd.DataFrame(recs).to_csv(EVAL/"unseen_predictions.csv", index=False); display(pd.DataFrame(recs))
    for p in sorted(out.glob("*"))[:8]: plt.figure(figsize=(6, 5)); plt.imshow(plt.imread(p)); plt.axis("off"); plt.title(p.name); plt.show()

## 8. FAVED integration: review, unified output, RAG hand-off
**What the FAVED README says** (I read the README; the backend source is not in this notebook):
- Vision: YOLOv11 transfer-learning detector, **7 classes**: cocoa Black Pod, Frosty Pod, Mirid; maize Common Rust, Gray Leaf Spot; plus healthy. Reported test P 86%, R 86%, mAP@50 92%, mAP@50-95 74%.
- Backend reads `DETECTION_MODEL_PATH`, `DOCUMENTS_FOLDER_PATH` (CROP_DISEASES knowledge docs), `LLM_MODEL_PATH` (N-ATLAS GGUF), `CACHE_PATH`; async job API (`/diagnose/start`, `/diagnose/status/{job_id}`); English + Hausa.
- **Overlap to resolve:** FAVED already has "Mirid" and our pest model has "Mirid-bug". The unified output merges them into one finding (`cocoa_mirid`) with both evidence sources.
- Their numbers are not comparable to ours (different task and test set). Re-evaluate the FAVED model on the same kind of held-out images before claiming anything.

In [ ]:
if not Path("/content/FAVED").exists():
    !git clone -q https://github.com/HopeAda/FAVED---Farm-AI-Vision-and-Recommendation-for-Early-Disease-Detection /content/FAVED
!ls /content/FAVED /content/FAVED/backend
print(open("/content/FAVED/backend/.env.example").read() if Path("/content/FAVED/backend/.env.example").exists() else "no .env.example found")
if FAVED_DISEASE_MODEL.exists():
    dm = YOLO(str(FAVED_DISEASE_MODEL)); print("\nFAVED disease model task:", dm.task); print("FAVED classes:", dm.names)
else:
    print(f"\nCopy FAVED's model.pt to {FAVED_DISEASE_MODEL} (it is the file DETECTION_MODEL_PATH points to) and re-run this cell.")

### 8b. `unified_pipeline.py`: one call -> pests + diseases -> one JSON
Edit `DISEASE_OVERRIDES` after printing the FAVED class names above if the keyword rules don't match their exact labels.

In [ ]:
%%writefile /content/unified_pipeline.py
"""Unified pest + disease analysis for FAVED. One image in, one JSON out (schema 1.0)."""
import os, json
import numpy as np, cv2
from ultralytics import YOLO

PEST_META = {  # model label -> (crop, knowledge-base key)
    "Large cutworm": ("maize", "large_cutworm"), "Yellow cutworm": ("maize", "yellow_cutworm"),
    "Corn borer": ("maize", "corn_borer"), "Army worm": ("maize", "army_worm"), "Aphids": ("maize", "maize_aphids"),
    "Pod-borer": ("cocoa", "cocoa_pod_borer"), "Weevil": ("cocoa", "cocoa_weevil"),
    "Mirid-bug": ("cocoa", "cocoa_mirid"), "Mealybug": ("cocoa", "cocoa_mealybug"),
}
_reg = os.getenv("PEST_REGISTRY_PATH")            # optional: extra crops added via class_registry.json
if _reg and os.path.exists(_reg):
    PEST_META.update({k: tuple(v) for k, v in json.load(open(_reg)).get("pest_meta", {}).items()})

DISEASE_RULES = [("black", "cocoa", "cocoa_black_pod", "disease"), ("frosty", "cocoa", "cocoa_frosty_pod", "disease"),
                 ("mirid", "cocoa", "cocoa_mirid", "pest"), ("rust", "maize", "maize_common_rust", "disease"),
                 ("gray", "maize", "maize_gray_leaf_spot", "disease"), ("grey", "maize", "maize_gray_leaf_spot", "disease")]
DISEASE_OVERRIDES = {}   # exact FAVED class name -> (type, crop, label_key)

def classify_disease_name(name):
    if name in DISEASE_OVERRIDES: return DISEASE_OVERRIDES[name]
    n = name.lower().replace("_", " ").replace("-", " ")
    if "healthy" in n:
        return ("healthy", "cocoa" if "cocoa" in n else "maize" if ("maize" in n or "corn" in n) else None, "healthy")
    for kw, crop, key, typ in DISEASE_RULES:
        if kw in n: return (typ, crop, key)
    return ("disease", None, n.strip().replace(" ", "_"))

class UnifiedDetector:
    def __init__(self, pest_model_path, disease_model_path=None, pest_conf=0.25, disease_conf=0.25, imgsz=640, review_conf=0.45):
        self.pest = YOLO(pest_model_path)
        self.disease = YOLO(disease_model_path) if disease_model_path else None
        self.pest_conf, self.disease_conf, self.imgsz, self.review_conf = pest_conf, disease_conf, imgsz, review_conf

    @classmethod
    def from_env(cls, **kw):
        return cls(os.environ["PEST_MODEL_PATH"], os.getenv("DETECTION_MODEL_PATH"), **kw)

    def _detections(self, model, source, image):
        conf = self.pest_conf if source == "pest_model" else self.disease_conf
        r = model.predict(image, conf=conf, imgsz=self.imgsz, iou=0.6, verbose=False)[0]
        h, w = r.orig_shape; out = []
        def meta(name):
            if source == "pest_model":
                crop, key = PEST_META.get(name, (None, name.lower().replace(" ", "_"))); return "pest", crop, key
            return classify_disease_name(name)
        if r.boxes is not None and len(r.boxes):
            for b in r.boxes:
                name = model.names[int(b.cls)]; typ, crop, key = meta(name)
                x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
                out.append(dict(type=typ, label=name, label_key=key, crop=crop, confidence=round(float(b.conf), 4),
                                bbox_xyxy=[round(v, 1) for v in (x1, y1, x2, y2)],
                                bbox_xywhn=[round(v, 4) for v in ((x1+x2)/2/w, (y1+y2)/2/h, (x2-x1)/w, (y2-y1)/h)], source_model=source))
        elif getattr(r, "probs", None) is not None:      # classifier-style disease model
            c = int(r.probs.top1); name = model.names[c]; typ, crop, key = meta(name)
            out.append(dict(type=typ, label=name, label_key=key, crop=crop, confidence=round(float(r.probs.top1conf), 4),
                            bbox_xyxy=None, bbox_xywhn=None, source_model=source))
        return out

    def analyze(self, image, language="en"):
        dets = self._detections(self.pest, "pest_model", image)
        if self.disease is not None: dets += self._detections(self.disease, "disease_model", image)
        findings = {}
        for d in dets:
            if d["type"] == "healthy": continue
            f = findings.setdefault(d["label_key"], dict(type=d["type"], label=d["label"], label_key=d["label_key"], crop=d["crop"],
                                                         count=0, max_confidence=0.0, evidence=set()))
            f["count"] += 1; f["max_confidence"] = max(f["max_confidence"], d["confidence"]); f["evidence"].add(d["source_model"])
        fl = sorted(findings.values(), key=lambda f: -f["max_confidence"])
        for f in fl: f["evidence"] = sorted(f["evidence"])
        votes = {}
        for d in dets:
            if d["crop"]: votes[d["crop"]] = votes.get(d["crop"], 0) + d["confidence"]
        crop = max(votes, key=votes.get) if votes else None
        pests = [f for f in fl if f["type"] == "pest"]; dis = [f for f in fl if f["type"] == "disease"]
        healthy = any(d["type"] == "healthy" for d in dets)
        status = "pest_and_disease" if pests and dis else "pest" if pests else "disease" if dis else "healthy" if healthy else "no_detection"
        warnings = []
        if status == "no_detection": warnings.append("Nothing detected. The crop or pest may be outside coverage, or the photo is unclear. Retake a close, well-lit photo.")
        elif fl and fl[0]["max_confidence"] < self.review_conf: warnings.append("Low confidence. Retake the photo (close-up, good light, one affected area) before acting.")
        queries = [dict(text=f"{f['label']} on {f['crop'] or crop or 'crop'}: how to identify, control and prevent it for smallholder farmers in Nigeria",
                        filters=dict(crop=f["crop"] or crop, category=f["type"], label_key=f["label_key"]), language=language) for f in fl[:3]]
        return dict(schema_version="1.0", crop=crop, status=status, primary_finding=(fl[0] if fl else None),
                    n_pests=len(pests), n_diseases=len(dis), findings=fl, detections=dets, warnings=warnings,
                    rag=dict(language=language, queries=queries))

def annotate(image_path, result):
    im = cv2.imread(image_path)
    for d in result["detections"]:
        if d["bbox_xyxy"] is None: continue
        x1, y1, x2, y2 = map(int, d["bbox_xyxy"]); col = (0, 0, 255) if d["type"] == "pest" else (0, 140, 255) if d["type"] == "disease" else (0, 200, 0)
        cv2.rectangle(im, (x1, y1), (x2, y2), col, 3); cv2.putText(im, f"{d['type']}:{d['label']} {d['confidence']:.2f}", (x1, max(20, y1-6)), 0, 0.7, col, 2)
    return im

if __name__ == "__main__":
    import sys
    print(json.dumps(UnifiedDetector.from_env().analyze(sys.argv[1]), indent=2))

In [ ]:
shutil.copy2("/content/unified_pipeline.py", ROOT/"unified_pipeline.py")
sys.path.insert(0, "/content"); import importlib, unified_pipeline; importlib.reload(unified_pipeline)
from unified_pipeline import UnifiedDetector, annotate
det = UnifiedDetector(str(FINAL/"pest_best.pt"), str(FAVED_DISEASE_MODEL) if FAVED_DISEASE_MODEL.exists() else None,
                      pest_conf=OPT_CONF, disease_conf=0.25, imgsz=int(card["imgsz"]))
sample = test_imgs[0]; result = det.analyze(str(sample)); print(json.dumps(result, indent=2)[:2500])
plt.figure(figsize=(7, 6)); plt.imshow(cv2.cvtColor(annotate(str(sample), result), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
(EVAL/"unified_sample_output.json").write_text(json.dumps(result, indent=2))

### 8c. Backend change (adapt names to the real `backend/app.py`)
I did not read FAVED's backend source, so treat this as the shape of the change:
```python
# backend/app.py
from unified_pipeline import UnifiedDetector
detector = UnifiedDetector.from_env()          # needs PEST_MODEL_PATH (new) + DETECTION_MODEL_PATH (existing) in .env
# inside the diagnose job, replace the single-model call:
result = detector.analyze(image_path, language=lang)
# result["rag"]["queries"] -> your existing retrieval over DOCUMENTS_FOLDER_PATH, then the N-ATLAS prompt
# result["findings"], result["status"], result["warnings"] -> JSON the React frontend renders
```
Add `PEST_MODEL_PATH=/path/to/pest_best.pt` to `.env.example`. The frontend's `ResultType` needs a `type: "pest" | "disease"` field and a list of findings instead of one label.

### 8d. RAG layer: knowledge base + retrieval + grounded prompt
The brief asks for retrieval, not hard-coded answers. Below: a knowledge-base scaffold keyed by `label_key`, a metadata-filtered retriever, and a prompt builder that **only answers from retrieved text** and falls back to "see an extension officer" when the KB is empty. The KB files are **templates**: the RAG team must fill them from citable sources (e.g. FAO, IITA, CABI, ICCO, Nigerian extension services) and record the source in each file. I did not invent agronomy content, because wrong pesticide advice can hurt farmers.

In [ ]:
from unified_pipeline import PEST_META
KB = ROOT/"knowledge_base"/"pests"; KB.mkdir(parents=True, exist_ok=True)
SECTIONS = ["Identification", "Damage and symptoms", "Conditions that favour it", "Cultural and prevention practices",
            "Biological and low-risk controls", "Chemical control (registered products only, label rates, protective equipment)",
            "When to contact an extension officer", "Sources"]
for label, (crop, key) in PEST_META.items():
    f = KB/f"{key}.md"
    if f.exists(): continue
    f.write_text(f"---\ncrop: {crop}\ncategory: pest\nlabel_key: {key}\ndisplay_name: {label}\nstatus: TODO\n---\n\n"
                 + "\n\n".join(f"## {s}\nTODO (RAG team): fill from a cited source." for s in SECTIONS) + "\n")
print(len(list(KB.glob("*.md"))), "KB templates in", KB)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_chunks(kb_root=ROOT/"knowledge_base"):
    chunks = []
    for md_file in kb_root.rglob("*.md"):
        head, _, body = md_file.read_text().partition("\n---\n")
        meta = dict(l.split(": ", 1) for l in head.replace("---\n", "", 1).splitlines() if ": " in l)
        for sec in body.split("\n## ")[1:] if "\n## " in body else []:
            title, _, text = sec.partition("\n")
            if "TODO" in text or not text.strip(): continue            # never retrieve unfilled templates
            chunks.append(dict(meta=meta, section=title.strip(), text=text.strip()))
    return chunks

def retrieve(query, filters, k=4):
    chunks = [c for c in load_chunks() if c["meta"].get("label_key") == filters.get("label_key")]
    if not chunks: return []
    vec = TfidfVectorizer(stop_words="english").fit([c["text"] for c in chunks] + [query["text"]])
    sims = cosine_similarity(vec.transform([query["text"]]), vec.transform([c["text"] for c in chunks]))[0]
    return [chunks[i] for i in sims.argsort()[::-1][:k]]      # RAG team: swap TF-IDF for embeddings

def build_llm_prompt(result, language="en"):
    ctx, missing = [], []
    for q in result["rag"]["queries"]:
        got = retrieve(q, q["filters"])
        (ctx.extend if got else missing.append)(got if got else q["filters"]["label_key"])
    lang = "Hausa" if language == "ha" else "English"
    return (f"You advise smallholder farmers. Answer in {lang}, in plain simple language.\n"
            "Use ONLY the context below. If the context does not cover something, say so; do not guess doses, products or rates.\n"
            "Always tell the farmer to follow product labels, wear protective equipment, and contact a local extension officer if unsure.\n"
            f"Detected: {[(f['label'], f['type'], round(f['max_confidence'], 2)) for f in result['findings']]} on {result['crop']}.\n"
            f"Warnings: {result['warnings']}\n\nCONTEXT:\n" + "\n\n".join(f"[{c['meta'].get('display_name')} / {c['section']}]\n{c['text']}" for c in ctx)
            + (f"\n\nNo verified guidance available for: {missing}. Tell the farmer to consult an extension officer." if missing else ""))

print(build_llm_prompt(result)[:1500])

## 9. Adding more crops later (class registry)
`merge_dataset` appends any YOLO-format pest dataset as **new class IDs** (existing IDs never change, so old labels stay valid), copies images with a crop prefix, rewrites `data.yaml`, and stores the label-to-crop map in `class_registry.json` (the pipeline reads it through `PEST_REGISTRY_PATH`). Then re-run Sections 2-8.

**Coverage gaps worth closing for Nigeria** (check with an agronomist; these are my reading of the class lists, not a verified survey):
- Maize: IP102's "army worm" is not the same as **fall armyworm** (*Spodoptera frugiperda*), a major maize pest in West Africa; stem borers also matter. Neither is clearly in the data yet.
- Cocoa: **pod-borer** (*Conopomorpha cramerella*) is, as far as I know, a Southeast Asian pest, so confirm it is relevant for your users. Mirids and mealybugs are the more relevant cocoa insects here.
- Other crops (cassava, yam, tomato, cowpea, rice, sorghum): search Roboflow Universe / Kaggle, and **check each dataset's licence** before using it.

In [ ]:
REGISTRY = ROOT/"class_registry.json"
def merge_dataset(src_root, crop, name_map, dst=DATA):
    """src_root: YOLO dataset (train/valid|val/test, images+labels). name_map: {source_class_id: 'Pest name'}."""
    cfg = yaml.safe_load((dst/"data.yaml").read_text()); names = {int(k): v for k, v in cfg["names"].items()}
    reg = json.loads(REGISTRY.read_text()) if REGISTRY.exists() else {"pest_meta": {}}
    new_id = {}
    for sid, nm in name_map.items():
        if nm not in names.values(): names[max(names) + 1] = nm
        new_id[int(sid)] = [k for k, v in names.items() if v == nm][0]
        reg["pest_meta"][nm] = [crop, f"{crop}_{re.sub('[^a-z0-9]+', '_', nm.lower()).strip('_')}"]
    for s_src, s_dst in [("train", "train"), ("valid", "valid"), ("val", "valid"), ("test", "test")]:
        if not (Path(src_root)/s_src/"images").exists(): continue
        for img in (Path(src_root)/s_src/"images").iterdir():
            lbl = Path(src_root)/s_src/"labels"/(img.stem + ".txt")
            if img.suffix.lower() not in IMG_EXT or not lbl.exists(): continue
            lines = [f"{new_id[int(float(p[0]))]} {' '.join(p[1:5])}" for p in (l.split() for l in lbl.read_text().splitlines()) if len(p) >= 5 and int(float(p[0])) in new_id]
            if not lines: continue
            shutil.copy2(img, dst/s_dst/"images"/f"{crop}_{img.name}"); (dst/s_dst/"labels"/f"{crop}_{img.stem}.txt").write_text("\n".join(lines) + "\n")
    cfg["names"], cfg["nc"] = names, len(names); (dst/"data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    REGISTRY.write_text(json.dumps(reg, indent=2)); print("classes now:", names)
# Example (after downloading a YOLO-format dataset):
# merge_dataset("/content/FallArmyworm", "maize", {0: "Fall armyworm"})
# NOTE: new crop files must not start with "cocoa_"; the audit's crop split in this notebook keys on that prefix.
# NOTE: leak-check any new dataset (Section 2) before training, and update NAMES/CROP_OF at the top of this notebook.

## 10. Auto-generated report + zip of everything

In [ ]:
def tbl(df):
    try: return df.round(3).to_markdown(index=False)
    except Exception: return df.round(3).to_string(index=False)
rep = f"""# Group 4 Pest Detection: ML report
Model: YOLO11 ({best_name}), selected on validation mAP@50-95. Test = held-out, leak-safe split.
Classes: {', '.join(f'{i}:{n}' for i, n in enumerate(NAMES))}

## Overall test metrics
{json.dumps({k: round(v, 4) for k, v in overall.items()})}

## Experiment comparison (validation)
{tbl(comp)}

## Per-class test metrics
{tbl(per_class)}

## Per-crop test metrics
{tbl(crop_tbl)}

## Error analysis at conf={OPT_CONF:.2f}
{tbl(err_tbl)}

Top confusions: {conf_pairs.most_common(5)}

## Robustness (mAP)
{tbl(rob)}

## Known limits
- Cocoa test set is small; treat per-class cocoa scores as indicative.
- Metrics are on curated dataset images, not Nigerian field photos. Collect and label field photos for a true real-world score.
- FAVED disease model was not re-evaluated here unless you ran it on a labelled set.
"""
(EVAL/"ML_REPORT.md").write_text(rep); print(rep[:1800])
shutil.make_archive(str(ROOT/"Group4_deliverables"), "zip", EVAL)
print("\nDeliverables on Drive:", ROOT)